In [1]:
import json
from pathlib import Path
from langchain_core.documents import Document


data_path = '/Users/yuhngtan/sandbox/improvement/data'

def load_documents(data_path: str):
    documents = []

    for json_file in Path(data_path).glob("**/*.json"):
        try:
            data = json.loads(json_file.read_text(encoding="utf-8"))
            doc_meta = {
                "source": str(json_file),
                "document_id": data.get("document_id"),
                "document_type": data.get("document_type"),
                **data.get("metadata", {}),  # candidate_id, job_title, company, ...
            }

            for section in data.get("sections", []):
                section_meta = {"section_id": section.get("section_id"), "section_name": section.get("section_name")}
                
                # a section has either its own "content" or a list of "entries" (jobs, projects), each with "content"
                for item in section.get("entries", [section]):
                    extra = {k: v for k, v in item.items() if k not in ("content", "section_id", "section_name")}
                    documents.append(Document(
                        page_content=item["content"],
                        metadata={**doc_meta, **section_meta, **extra},
                    ))

        except Exception as e:
            print(f"[ERROR] {json_file}: {e}")

    return documents


if __name__ == "__main__":
    docs = load_documents(data_path)

    cv_docs = [doc for doc in docs if doc.metadata["document_type"] == "resume"]
    jd_docs = [doc for doc in docs if doc.metadata["document_type"] == "job_description"]

    print(f"Loaded {len(docs)} documents ({len(cv_docs)} CV, {len(jd_docs)} JD).")
    print("Example document:", docs[0] if docs else None)

Loaded 13 documents (9 CV, 4 JD).
Example document: page_content='Data Science and Business Analytics undergraduate with experience in data engineering, analytics, and AI/ML application development. Skilled in Python, SQL, PySpark, Databricks, and cloud data platforms. Experienced in developing data pipelines, building analytical solutions, and deploying AI-powered applications.' metadata={'source': '/Users/yuhngtan/sandbox/improvement/data/cv.json', 'document_id': 'CV_001', 'document_type': 'resume', 'candidate_id': 'CANDIDATE_001', 'target_role': 'Data Engineer', 'section_id': 'CV_SEC_001', 'section_name': 'professional_summary'}


In [ ]:
import torch

class RetrievalPipeline:
    def __init__(self, top_k=5):
        self.top_k = top_k

    def retrieve(self, cv_chunks, jd_chunks, cv_embeddings, jd_embeddings):
        # normalized embeddings → dot product = cosine similarity; shape (n_jd, n_cv)
        similarity_matrix = jd_embeddings @ cv_embeddings.T

        top_k = min(self.top_k, len(cv_chunks))  # torch.topk errors if k > n_cv
        top_scores, top_indices = torch.topk(similarity_matrix, k=top_k, dim=1)  # sorted best first

        results = []
        for jd_idx, jd_chunk in enumerate(jd_chunks):
            for score, cv_idx in zip(top_scores[jd_idx].tolist(), top_indices[jd_idx].tolist()):
                results.append({
                    "jd_chunk": jd_chunk,
                    "cv_chunk": cv_chunks[cv_idx],
                    "bi_encoder_score": score,
                })
        return results

if __name__ == "__main__":
    from improvement.src.data_loader import load_documents, data_path
    from improvement.src.embedding import EmbeddingPipeline

    emb_pipe = EmbeddingPipeline()
    cv_chunks, jd_chunks = emb_pipe.chunk_documents(load_documents(data_path))
    cv_embeddings, jd_embeddings = emb_pipe.embed_chunks(cv_chunks, jd_chunks)

    results = RetrievalPipeline(top_k=3).retrieve(cv_chunks, jd_chunks, cv_embeddings, jd_embeddings)
    for r in results:
        jd_meta, cv_meta = r["jd_chunk"].page_content, r["cv_chunk"].page_content
        print(f"{r['bi_encoder_score']:.3f}  {jd_meta} -> {cv_meta}")